# Clinical Sample Selection and Compile L3 AIFI cell types  

Subset L3 AIFI cell type objects based on 141 selected clinical samples

Concatenate L3 cell type objects together to compiled anndata object

## Load libraries

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from datetime import date
import hisepy
import os
import pandas as pd
import re
import scanpy as sc
import scanpy.external as sce
import anndata as ad
import glob
import random


In [2]:
out_dir = 'output_sample_selection'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

## Helper functions

These make it a bit simpler to cache and read in files from HISE

In [3]:
def cache_uuid_path(uuid):
    cache_path = '/home/jupyter/cache/{u}'.format(u = uuid)
    if not os.path.isdir(cache_path):
        hise_res = hisepy.reader.cache_files([uuid])
    filename = os.listdir(cache_path)[0]
    cache_file = '{p}/{f}'.format(p = cache_path, f = filename)
    return cache_file

In [4]:
def read_parquet_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = pd.read_parquet(cache_file)
    return res

In [5]:
def sort_adata_uuid(uuid, sort_cols = ['AIFI_L2', 'sample.sampleKitGuid']):
    cache_file = cache_uuid_path(uuid)
    adata = sc.read_h5ad(cache_file)
    obs = adata.obs
    obs = obs.sort_values(sort_cols)
    adata = adata[obs.index]
    adata.write_h5ad(cache_file)

This function will enable us to connect to our .h5ad files without loading the entire thing into memory. We'll then load only the cells that we want for each cell class to assemble them for writing. This should save us some overhead as we do our subsetting.

In [6]:
def read_adata_backed_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = sc.read_h5ad(cache_file, backed = 'r')
    return res

This function will remove Ig-related genes, which is recommended by Marla Glass for analysis of B cell subtypes

In [7]:
def remove_ig_genes(adata):
    igl_genes = [gene for gene in adata.var_names if gene.startswith("IGL")]
    igk_genes = [gene for gene in adata.var_names if gene.startswith("IGK")]
    ighc_genes = [gene for gene in adata.var_names if gene.startswith("IGH")]
    exl_genes = igl_genes + igk_genes + ighc_genes

    filtered_genes = [gene for gene in adata.var_names if gene not in exl_genes]
    adata = adata[:, filtered_genes]

    return adata

This function will apply a standard normalization, nearest neighbors, clustering, and UMAP process to our cell subsets:

In [8]:
def process_adata(adata, resolution = 2):
    
    # Keep a copy of the raw data
    adata = adata.raw.to_adata()
    adata.raw = adata

    print('Normalizing', end = "; ")
    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    print('Finding HVGs', end = "; ")
    # Restrict downstream steps to variable genes
    sc.pp.highly_variable_genes(adata)
    adata = adata[:, adata.var_names[adata.var['highly_variable']]].copy()

    print('Scaling', end = "; ")
    # Scale variable genes
    sc.pp.scale(adata)

    print('PCA', end = "; ")
    # Run PCA
    sc.tl.pca(adata, svd_solver = 'arpack')
    
    print('Neighbors', end = "; ")
    # Find nearest neighbors
    sc.pp.neighbors(
        adata, 
        n_neighbors = 50,
        n_pcs = 30
    )

    print('Leiden', end = "; ")
    # Find clusters
    sc.tl.leiden(
        adata, 
        resolution = resolution, 
        key_added = 'leiden_{r}'.format(r = resolution),
        n_iterations = 2
    )

    print('UMAP', end = "; ")
    # Run UMAP
    sc.tl.umap(adata, min_dist = 0.05)
    
    print('Renormalizing')
    adata = adata.raw.to_adata()
    adata.raw = adata

    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    
    return adata

In [9]:
def format_cell_type(cell_type):
    cell_type = re.sub('\\+', 'pos', cell_type)
    cell_type = re.sub('-', 'neg', cell_type)
    cell_type = re.sub(' ', '_', cell_type)
    return cell_type

In [10]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

In [11]:
# make a function to find files
def get_filepaths_with_glob(root_path: str, file_regex: str):
    return glob.glob(os.path.join(root_path, file_regex))

In [12]:
# Define a function to extract the desired substring using regex
def extract_substring(path, pattern = r'preRA_cluster_harmonize_(.*?)_\d{4}-\d{2}-\d{2}\.h5ad'):
    #pattern = r'preRA_cluster_harmonize_(.*?)_\d{4}-\d{2}-\d{2}\.h5ad'
    match = re.search(pattern, path)
    if match:
        return match.group(1)
    return None

In [13]:
def read_anndata_files(file_tuples):
    """
    Read Anndata objects from H5AD files and store them in a dictionary with custom names.

    Parameters:
        file_tuples (list of tuples): List of tuples where each tuple contains filename and desired name.

    Returns:
        dict: Dictionary containing Anndata objects with custom names.
    """
    anndata_dict = {}
    for filename, name in file_tuples:
        anndata_obj = anndata.read_h5ad(filename)
        anndata_dict[name] = anndata_obj
    return anndata_dict

In [14]:
def reformat_cell_type(cell_type):
    '''convert cell type names read in from file back to original L3 labels'''
    cell_type = re.sub('pos', '+', cell_type)
    cell_type = re.sub('neg', '-', cell_type)
    cell_type = re.sub('_',' ', cell_type)
    return cell_type

## Identify files for use in HISE

In [50]:
search_id = 'molybdenum-niobium-krypton'

Retrieve files stored in our HISE project store

In [51]:
ps_df = hisepy.list_files_in_project_store('UCSDCU_Y4')
ps_df = ps_df[['id', 'name']]

Filter for files from the previous notebook using our search_id

In [52]:
search_df = ps_df[ps_df['name'].str.contains(search_id)]
search_df = search_df.sort_values('name')

In [53]:
h5ad_df = search_df[search_df['name'].str.contains('.h5ad')]

In [54]:
h5ad_df

,id,name
529,2558b706-8cac-41c6-bd21-8493d514762d,molybdenum-niobium-krypton/preRA_deepcleaned_A...
544,61222f54-586a-44a8-8aed-61576c1ab381,molybdenum-niobium-krypton/preRA_deepcleaned_A...
487,fe955a14-15e0-4917-9457-814bde477e85,molybdenum-niobium-krypton/preRA_deepcleaned_A...
516,415df319-8ecd-440b-9186-30e5b73f4d7c,molybdenum-niobium-krypton/preRA_deepcleaned_B...
518,62f3b6b8-ae35-47d4-832b-5c91e0b0257f,molybdenum-niobium-krypton/preRA_deepcleaned_C...
...,...,...
551,cd741edc-3e77-4b74-89bc-273e940175a8,molybdenum-niobium-krypton/preRA_deepcleaned_S...
492,37f8a343-d026-4f57-9b84-04ee83d171c3,molybdenum-niobium-krypton/preRA_deepcleaned_T...
536,71e2f493-a14e-4065-b771-fef348a5059b,molybdenum-niobium-krypton/preRA_deepcleaned_T...
502,86eee4ad-7e70-4b08-a2d8-d6c537205ecd,molybdenum-niobium-krypton/preRA_deepcleaned_c...


In [55]:
h5ad_uuids = {}
for i in range(h5ad_df.shape[0]):
    fn = h5ad_df['name'].tolist()[i]
    group_name = re.sub('.+pbmc_', '', fn)
    group_name = re.sub('_init.+', '', group_name)
    h5ad_uuids[group_name] = h5ad_df['id'].tolist()[i]

In [56]:
h5ad_uuids

{'molybdenum-niobium-krypton/preRA_deepcleaned_ASDC_2024-06-13.h5ad': '2558b706-8cac-41c6-bd21-8493d514762d',
 'molybdenum-niobium-krypton/preRA_deepcleaned_Activated_memory_B_cell_2024-06-13.h5ad': '61222f54-586a-44a8-8aed-61576c1ab381',
 'molybdenum-niobium-krypton/preRA_deepcleaned_Adaptive_NK_cell_2024-06-13.h5ad': 'fe955a14-15e0-4917-9457-814bde477e85',
 'molybdenum-niobium-krypton/preRA_deepcleaned_BaEoMaP_cell_2024-06-13.h5ad': '415df319-8ecd-440b-9186-30e5b73f4d7c',
 'molybdenum-niobium-krypton/preRA_deepcleaned_C1Qpos_CD16_monocyte_2024-06-13.h5ad': '62f3b6b8-ae35-47d4-832b-5c91e0b0257f',
 'molybdenum-niobium-krypton/preRA_deepcleaned_CD14pos_cDC2_2024-06-13.h5ad': 'bd35e899-3b6b-42ec-b7fd-503036ef4be8',
 'molybdenum-niobium-krypton/preRA_deepcleaned_CD27neg_effector_B_cell_2024-06-13.h5ad': 'fcf2b06d-6fdf-4846-9793-cc2edb92c8ce',
 'molybdenum-niobium-krypton/preRA_deepcleaned_CD27pos_effector_B_cell_2024-06-13.h5ad': '8d3dbec0-2179-493a-bf4a-5036a7adae10',
 'molybdenum-niobiu

In [57]:
len(h5ad_uuids)

71

## Download and sort .h5ad files

Sorting the .h5ad files by AIFI_L2 will make reading each cell type much faster by placing cells of the same type next to each other in the sparse matrix.

In [25]:

# run once
for uuid in h5ad_uuids.values():
    sort_adata_uuid(uuid, sort_cols = ['AIFI_L3'])

downloading fileID: 2558b706-8cac-41c6-bd21-8493d514762d
Files have been successfully downloaded!
downloading fileID: 61222f54-586a-44a8-8aed-61576c1ab381
Files have been successfully downloaded!
downloading fileID: fe955a14-15e0-4917-9457-814bde477e85
Files have been successfully downloaded!
downloading fileID: 415df319-8ecd-440b-9186-30e5b73f4d7c
Files have been successfully downloaded!
downloading fileID: 62f3b6b8-ae35-47d4-832b-5c91e0b0257f
Files have been successfully downloaded!
downloading fileID: bd35e899-3b6b-42ec-b7fd-503036ef4be8
Files have been successfully downloaded!
downloading fileID: fcf2b06d-6fdf-4846-9793-cc2edb92c8ce
Files have been successfully downloaded!
downloading fileID: 8d3dbec0-2179-493a-bf4a-5036a7adae10
Files have been successfully downloaded!
downloading fileID: e5b5ca1d-9bb3-4afe-ad41-efc778ed292d
Files have been successfully downloaded!
downloading fileID: 75a5fa47-601b-49d1-af1c-b4da9b08a875
Files have been successfully downloaded!
downloading fileID: 

## Open connections to .h5ad files

Now that they're sorted, we can open these files with on-disk backing so we don't have to read the entire file at once.

## Process files

In [15]:
input_path = "/home/jupyter/cache-06-deepclean/**/"

In [16]:
### read in processed leiden adata
filenames = get_filepaths_with_glob(input_path, "preRA_deepcleaned_*.h5ad")  
filenames[:5]

['/home/jupyter/cache-06-deepclean/5819511f-de3a-4dc6-a794-fc1e819bc5aa/preRA_deepcleaned_ISGpos_naive_CD4_T_cell_2024-06-13.h5ad',
 '/home/jupyter/cache-06-deepclean/8d3dbec0-2179-493a-bf4a-5036a7adae10/preRA_deepcleaned_CD27pos_effector_B_cell_2024-06-13.h5ad',
 '/home/jupyter/cache-06-deepclean/91459173-8cf0-401e-b7df-7adee891f731/preRA_deepcleaned_HLAnegDRhi_cDC2_2024-06-13.h5ad',
 '/home/jupyter/cache-06-deepclean/fe955a14-15e0-4917-9457-814bde477e85/preRA_deepcleaned_Adaptive_NK_cell_2024-06-13.h5ad',
 '/home/jupyter/cache-06-deepclean/a7e61f28-74c5-45a7-9bfe-dd89524be3e4/preRA_deepcleaned_CM_CD4_T_cell_2024-06-13.h5ad']

In [17]:
### extract cell types
# Apply the function to each filename in the list using list comprehension
cell_types = [extract_substring(path,pattern = r'preRA_deepcleaned_(.*?)_\d{4}-\d{2}-\d{2}\.h5ad') for path in filenames]
cell_types[:5]

['ISGpos_naive_CD4_T_cell',
 'CD27pos_effector_B_cell',
 'HLAnegDRhi_cDC2',
 'Adaptive_NK_cell',
 'CM_CD4_T_cell']

In [18]:
file_dict = dict(zip(cell_types, filenames))
list(file_dict.items())[:10]

[('ISGpos_naive_CD4_T_cell',
  '/home/jupyter/cache-06-deepclean/5819511f-de3a-4dc6-a794-fc1e819bc5aa/preRA_deepcleaned_ISGpos_naive_CD4_T_cell_2024-06-13.h5ad'),
 ('CD27pos_effector_B_cell',
  '/home/jupyter/cache-06-deepclean/8d3dbec0-2179-493a-bf4a-5036a7adae10/preRA_deepcleaned_CD27pos_effector_B_cell_2024-06-13.h5ad'),
 ('HLAnegDRhi_cDC2',
  '/home/jupyter/cache-06-deepclean/91459173-8cf0-401e-b7df-7adee891f731/preRA_deepcleaned_HLAnegDRhi_cDC2_2024-06-13.h5ad'),
 ('Adaptive_NK_cell',
  '/home/jupyter/cache-06-deepclean/fe955a14-15e0-4917-9457-814bde477e85/preRA_deepcleaned_Adaptive_NK_cell_2024-06-13.h5ad'),
 ('CM_CD4_T_cell',
  '/home/jupyter/cache-06-deepclean/a7e61f28-74c5-45a7-9bfe-dd89524be3e4/preRA_deepcleaned_CM_CD4_T_cell_2024-06-13.h5ad'),
 ('Erythrocyte',
  '/home/jupyter/cache-06-deepclean/09eb5aa9-8e09-4b9e-bceb-ed61e025c35c/preRA_deepcleaned_Erythrocyte_2024-06-13.h5ad'),
 ('SOX4pos_naive_CD4_T_cell',
  '/home/jupyter/cache-06-deepclean/86b1d173-8b18-4de6-ae77-812a77

## Read in clinical metadata

In [26]:
### read in 144 pre-ra clinical samples
meta_sample_selection = pd.read_csv("../data/ALTRA_scRNA_combined_141_samples_info.csv")
meta_sample_selection

,lastUpdated,sample.id,sample.bridgingControl,sample.sampleKitGuid,sample.visitName,sample.visitDetails,sample.drawDate,sample.daysSinceFirstVisit,sample.diseaseStatesRecordedAtVisit,file.id,...,subject.biologicalSex,subject.birthYear,subject.ethnicity,subject.partnerCode,subject.race,subject.subjectGuid,cohort.cohortGuid,status,age,days_to_conversion
0,2023-11-09T02:52:35Z,85398587-1d17-4df0-8aab-3447c638b185,False,KT00070,Flu Year 1 Day 0,N/A - Flu-Series Timepoint Only,2019-10-01T00:00:00Z,0,Rheumatoid arthritis,1312850d-c89e-46b9-a1c1-1c3a8518e7b9,...,Male,1943,Non-Hispanic origin,CU,Caucasian,CU1001,CU1,ACPA+ at_risk,76,NaN
1,2023-11-09T02:52:35Z,0b9379d4-39c7-400e-ab24-dd91555145c1,False,KT00058,Flu Year 1 Day 0,N/A - Flu-Series Timepoint Only,2019-10-01T00:00:00Z,0,Rheumatoid arthritis,b5020ad1-139f-4c73-8651-56ddd4713742,...,Female,1941,Non-Hispanic origin,CU,Caucasian,CU1002,CU1,ACPA+ at_risk,78,NaN
2,2023-11-09T02:52:35Z,b606576c-0bad-4756-97b7-f275137c9d64,False,KT00057,Flu Year 1 Day 0,N/A - Flu-Series Timepoint Only,2019-10-01T00:00:00Z,0,Rheumatoid arthritis,1d9d0c7a-640d-43f9-8f49-a10e5f3da4b9,...,Female,1998,Non-Hispanic origin,CU,Caucasian,CU1003,CU1,ACPA+ at_risk,21,NaN
3,2023-11-09T02:52:35Z,3aafec3c-e5a5-4747-bcb5-06ba06b210c7,False,KT00077,Flu Year 1 Day 0,N/A - Flu-Series Timepoint Only,2019-10-01T00:00:00Z,0,Rheumatoid arthritis,34f7fb29-e412-4f92-8770-55f8e19d40fe,...,Female,1987,Non-Hispanic origin,CU,Caucasian,CU1004,CU1,ACPA+ at_risk,32,NaN
4,2023-11-09T02:52:35Z,b94bc5ba-31e0-45ab-a323-c30fbb318a8e,False,KT00060,Flu Year 1 Day 0,N/A - Flu-Series Timepoint Only,2019-11-01T00:00:00Z,0,Rheumatoid arthritis,b7f20e0b-b86a-4cb6-ab13-365e2504bda6,...,Female,1979,Non-Hispanic origin,CU,Caucasian,CU1005,CU1,ACPA+ at_risk,40,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,2023-11-09T02:52:35Z,60d00ab6-7c06-414a-8092-43dd56156621,False,KT03939,RA Year 4 Visit 2,N/A - stand-alone collection,2022-12-01T00:00:00Z,577,CCP3+ At Risk for RA,83a149aa-5025-4f80-a3a0-b88a4fd60be7,...,Female,1988,Non-Hispanic origin,CU,Caucasian,CU1052,CU1,pre,34,-140.0
137,2023-11-09T02:52:35Z,2bd26d24-a021-4cd2-82b1-3e8cffa5f7df,False,KT02968,Flu Year 1 Day 90,N/A - Flu-Series Timepoint Only,2022-02-01T00:00:00Z,276,Rheumatoid arthritis,82039365-4d2c-4dc6-8084-b2593d5fb8f4,...,Female,1988,Non-Hispanic origin,CU,Caucasian,CU1052,CU1,pre,34,-441.0
138,2023-11-09T02:52:35Z,d0d3603a-28d4-4003-b21e-e06bb68b5ec9,False,KT03932,RA Year 4 Visit 1,N/A - stand-alone collection,2022-08-01T00:00:00Z,458,Rheumatoid arthritis,a2829968-97f2-48c8-bc0a-2989850aebf1,...,Female,1988,Non-Hispanic origin,CU,Caucasian,CU1052,CU1,pre,34,-259.0
139,2023-11-09T02:52:35Z,eaa9a00c-5a22-4643-96de-12fe665afa82,False,KT00103,Flu Year 2 Stand-Alone,N/A - Flu-Series Timepoint Only,2020-07-01T00:00:00Z,158,Rheumatoid arthritis,5dd9a323-260c-4689-aebf-0b551ba9a553,...,Female,1984,Hispanic or Latino origin,SD,Other,SD1003,SD1,pre,36,-80.0


In [27]:
## check DTC is under -750
meta_sample_selection[['days_to_conversion']].describe()

,days_to_conversion
count,40.000000
mean,-358.000000
std,233.457842
min,-1035.000000
25%,-502.500000
50%,-330.500000
75%,-138.000000
max,-80.000000


In [28]:
meta_sample_selection.columns

Index(['lastUpdated', 'sample.id', 'sample.bridgingControl',
       'sample.sampleKitGuid', 'sample.visitName', 'sample.visitDetails',
       'sample.drawDate', 'sample.daysSinceFirstVisit',
       'sample.diseaseStatesRecordedAtVisit', 'file.id', 'file.name',
       'file.batchID', 'file.panel', 'file.pool', 'file.fileType',
       'file.userTags.details', 'file.userTags.group', 'file.userTags.name',
       'file.userTags.origin', 'file.userTags.other', 'file.userTags.version',
       'file.majorVersion', 'subject.id', 'subject.biologicalSex',
       'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode',
       'subject.race', 'subject.subjectGuid', 'cohort.cohortGuid', 'status',
       'age', 'days_to_conversion'],
      dtype='object')

In [31]:
meta_sample_selection = meta_sample_selection[[ 'sample.sampleKitGuid', 'subject.subjectGuid', 'status', 'age', 'days_to_conversion']]

meta_sample_selection

,sample.sampleKitGuid,subject.subjectGuid,status,age,days_to_conversion
0,KT00070,CU1001,ACPA+ at_risk,76,NaN
1,KT00058,CU1002,ACPA+ at_risk,78,NaN
2,KT00057,CU1003,ACPA+ at_risk,21,NaN
3,KT00077,CU1004,ACPA+ at_risk,32,NaN
4,KT00060,CU1005,ACPA+ at_risk,40,NaN
...,...,...,...,...,...
136,KT03939,CU1052,pre,34,-140.0
137,KT02968,CU1052,pre,34,-441.0
138,KT03932,CU1052,pre,34,-259.0
139,KT00103,SD1003,pre,36,-80.0


## Process Each Cell Types

In [32]:
### create unit test: cell type with no renaming, cell type with 1 rename and cell type with multiple rename
#subfile_dict = dict((k, file_dict[k]) for k in ('CD4_MAIT', 'ASDC', 'CM_CD4_T_cell'))
subfile_dict = dict((k, file_dict[k]) for k in ('CD4_MAIT', 'Core_naive_CD4_T_cell'))

subfile_dict

{'CD4_MAIT': '/home/jupyter/cache-06-deepclean/e5b5ca1d-9bb3-4afe-ad41-efc778ed292d/preRA_deepcleaned_CD4_MAIT_2024-06-13.h5ad',
 'Core_naive_CD4_T_cell': '/home/jupyter/cache-06-deepclean/7c347c97-7789-444d-bbef-af324f619ea1/preRA_deepcleaned_Core_naive_CD4_T_cell_2024-06-13.h5ad'}

In [33]:
cell_types_list = []
cell_types_list

[]

In [34]:
for label, file in file_dict.items():
    print("Key:", label)
    print("Value:", file)

    adata = sc.read_h5ad(file)

    # restore raw counts to X
    adata = adata.raw.to_adata()
    # delete raw layer
    adata.raw = None

    #print(adata)
    
    meta = adata.obs

    print("Unfiltered metadata shape:" + str(meta.shape))
    
    # Perform the merge
    meta2 = pd.merge(meta, meta_sample_selection, how='inner', on=['sample.sampleKitGuid'])
    
    print("Filtered metadata shape " + str(meta2.shape))
    
    print("Number of unique clinical sample selected: "+ str(len(meta2['sample.sampleKitGuid'].unique())) )
    
    # subset anndata by 
    adata_subset = adata[adata.obs['sample.sampleKitGuid'].isin(meta2['sample.sampleKitGuid'])]

    adata_subset_before = adata_subset
    
    #adata_subset.obs = meta2.set_index(adata_subset.obs.index) -- omit
    # adata_subset.obs = meta2.set_index(['barcodes'])
    print(adata_subset)
    
    # append to outfile list
    cell_types_list.append(adata_subset)

Key: ISGpos_naive_CD4_T_cell
Value: /home/jupyter/cache-06-deepclean/5819511f-de3a-4dc6-a794-fc1e819bc5aa/preRA_deepcleaned_ISGpos_naive_CD4_T_cell_2024-06-13.h5ad
Unfiltered metadata shape:(24358, 50)
Filtered metadata shape (9291, 54)
Number of unique clinical sample selected: 141
View of AnnData object with n_obs × n_vars = 9291 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_coun

In [35]:
cell_types_list[:3]

[View of AnnData object with n_obs × n_vars = 9291 × 33538
     obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden_harmony_2', 'doublets_manual', 'AIFI_L3_n

In [36]:
len(cell_types_list)

71

## Concatenate cell types into single object

In [37]:
# Concatenate all aligned AnnData objects in the list
adata_combined = ad.concat(cell_types_list, axis=0)

In [38]:
adata_combined

AnnData object with n_obs × n_vars = 2059581 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden_harmony_2', 'doublets_manual', 'AIFI_L3_new'
   

In [39]:
adata_combined.obs

,barcodes,batch_id,cell_name,cell_uuid,chip_id,hto_barcode,hto_category,n_genes,n_mito_umis,n_reads,...,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mito,log1p_total_counts_mito,pct_counts_mito,leiden_harmony_2,doublets_manual,AIFI_L3_new
barcodes,,,,,,,,,,,,,,,,,,,,,
6a185d4cc0a311ebb325a228a802157d,6a185d4cc0a311ebb325a228a802157d,B070,preroyal_supporting_sifaka,6a185d4cc0a311ebb325a228a802157d,B070-P2C2,AAGTATCGTTTCGCA,singlet,1668,136,21709,...,50.513377,64.208243,72.321041,82.386117,136,4.919981,1.966739,13,no,ISG+ naive CD4 T cell
ca5551b4dc5f11eca5893ed7e0e9bbf0,ca5551b4dc5f11eca5893ed7e0e9bbf0,B122,pyroxene_hapless_arrowworm,ca5551b4dc5f11eca5893ed7e0e9bbf0,B122-P2C2,ATTGACCCGCGTTAG,singlet,1512,159,14698,...,40.584150,54.513889,64.930556,78.921569,159,5.075174,3.247549,1,no,ISG+ naive CD4 T cell
e467d71076fe11eb934cc6c1afd85b40,e467d71076fe11eb934cc6c1afd85b40,B046,rubellite_devotional_quoll,e467d71076fe11eb934cc6c1afd85b40,B046-P2C3,ATTGACCCGCGTTAG,singlet,1582,93,17204,...,48.740490,62.519019,70.735418,81.707523,93,4.543295,1.572274,16,no,ISG+ naive CD4 T cell
b183f0c6dc5c11eca3e6166421532e92,b183f0c6dc5c11eca3e6166421532e92,B122,embattled_epicurean_sloth,b183f0c6dc5c11eca3e6166421532e92,B122-P2C2,AAATCTCTCAGGCTC,singlet,1706,256,26163,...,49.441497,64.744485,72.312203,82.113935,256,5.549076,3.574421,1,no,ISG+ naive CD4 T cell
ac2212a8c0a511eb92cb5ea76614d713,ac2212a8c0a511eb92cb5ea76614d713,B070,calcite_engrammic_insect,ac2212a8c0a511eb92cb5ea76614d713,B070-P2C3,AAGTATCGTTTCGCA,singlet,1777,57,21380,...,46.645993,59.343888,67.259670,78.194875,57,4.060443,0.930308,3,no,ISG+ naive CD4 T cell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
e263995ed92111eda28526bc5bccb9e5,e263995ed92111eda28526bc5bccb9e5,B140,impressive_nirvanic_nerka,e263995ed92111eda28526bc5bccb9e5,B140-P2C3,ATTGACCCGCGTTAG,singlet,3705,540,75409,...,35.745066,49.412140,60.562856,72.203011,540,6.293419,2.872799,10,no,ISG+ CD14 monocyte
f7e729ae424a11ecb1157ed7241a30b0,f7e729ae424a11ecb1157ed7241a30b0,B010,draughty_drifty_steed,f7e729ae424a11ecb1157ed7241a30b0,B010-P1C2,ATTGACCCGCGTTAG,singlet,2266,322,24217,...,34.939759,45.639702,56.454389,70.080321,322,5.777652,4.618474,3,no,ISG+ CD14 monocyte
f7e212e8424a11ecb1157ed7241a30b0,f7e212e8424a11ecb1157ed7241a30b0,B010,statutory_governable_crocodile,f7e212e8424a11ecb1157ed7241a30b0,B010-P1C2,ATTGACCCGCGTTAG,singlet,1918,161,20012,...,33.548815,45.566053,57.782198,72.998508,161,5.087596,2.668656,16,no,ISG+ CD14 monocyte


In [40]:
adata.obs

,barcodes,batch_id,cell_name,cell_uuid,chip_id,hto_barcode,hto_category,n_genes,n_mito_umis,n_reads,...,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mito,log1p_total_counts_mito,pct_counts_mito,leiden_harmony_2,doublets_manual,AIFI_L3_new
barcodes,,,,,,,,,,,,,,,,,,,,,
cd1dea26475e11eea33deebe6eaee26b,cd1dea26475e11eea33deebe6eaee26b,B149,sudden_waspish_hapuka,cd1dea26475e11eea33deebe6eaee26b,B149-P2C3,ATTGACCCGCGTTAG,singlet,2603,123,32311,...,31.019795,41.781508,52.468053,66.386870,123,4.820282,1.540967,9,no,ISG+ CD14 monocyte
150d810adb0b11ecbf282aed7cb8a617,150d810adb0b11ecbf282aed7cb8a617,B122,synarchist_crackable_pekingese,150d810adb0b11ecbf282aed7cb8a617,B122-P1C1,TTCCGCCTCTCTTTG,singlet,2350,300,20758,...,31.172145,41.197659,51.613387,66.276452,300,5.707110,4.502476,6,no,ISG+ CD14 monocyte
23ac8812770411eb97ab6e0733497e1b,23ac8812770411eb97ab6e0733497e1b,B046,dexterous_accurate_steer,23ac8812770411eb97ab6e0733497e1b,B046-P2C3,CTGTATGTCCGATTG,singlet,1812,29,16962,...,30.650155,42.063983,53.931889,70.175439,29,3.401197,0.598555,3,no,ISG+ CD14 monocyte
ef5d57341cb111eea7e0a2c3b96266dd,ef5d57341cb111eea7e0a2c3b96266dd,B167,subsequent_bloodshot_rodent,ef5d57341cb111eea7e0a2c3b96266dd,B167-P1C1,TAACGACCAGCCATA,singlet,2902,316,31856,...,31.309429,43.423138,54.744453,67.551506,316,5.758902,3.129952,7,no,ISG+ CD14 monocyte
6aa9f23e225d11eeb3a416d9d3689d72,6aa9f23e225d11eeb3a416d9d3689d72,B159,hypodermal_tiny_siamesecat,6aa9f23e225d11eeb3a416d9d3689d72,B159-P1C1,CAGTAGTCACGGTCA,singlet,2578,179,26626,...,28.103356,38.891849,49.866809,64.824188,179,5.192957,2.384124,7,no,ISG+ CD14 monocyte
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5768432c35e211ee8639569e6e3090a1,5768432c35e211ee8639569e6e3090a1,B133,brickish_radiation_ungulate,5768432c35e211ee8639569e6e3090a1,B133-P1C1,TAACGACCAGCCATA,singlet,2711,178,17957,...,29.031018,37.631377,47.513458,62.599334,178,5.187386,2.281466,6,no,ISG+ CD14 monocyte
fe60dc00287511ed81167a73bb3e8ffe,fe60dc00287511ed81167a73bb3e8ffe,B134,silty_raptorial_chick,fe60dc00287511ed81167a73bb3e8ffe,B134-P1C1,AGTAAGTTCAGCGTA,singlet,2391,116,37258,...,27.165118,37.383878,48.966137,65.507941,116,4.762174,1.738088,14,no,ISG+ CD14 monocyte
8ad422d0643b11ee89966abbdf1ae791,8ad422d0643b11ee89966abbdf1ae791,B177,childly_tangerine_megaraptor,8ad422d0643b11ee89966abbdf1ae791,B177-P1C1,AGTAAGTTCAGCGTA,singlet,3792,396,63731,...,29.517639,39.100072,48.920086,61.943844,396,5.983936,2.850972,12,no,ISG+ CD14 monocyte


In [42]:
adata

AnnData object with n_obs × n_vars = 164281 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden_harmony_2', 'doublets_manual', 'AIFI_L3_new'
    

In [41]:
#### test without randomonization

# get test data
joint_adata_test = adata_combined[adata_combined.obs['barcodes'].isin(adata.obs.index)].copy()
#adata_subset2 = adata_subset[adata_subset.obs['barcodes'].isin(random_barcodes)].copy()
joint_adata_test

AnnData object with n_obs × n_vars = 49159 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden_harmony_2', 'doublets_manual', 'AIFI_L3_new'
    o

In [43]:
cert_exprs = joint_adata_test.to_df()
cert_exprs.index=joint_adata_test.obs['barcodes']
cert_exprs = cert_exprs.sort_index()

In [44]:
cd4na_exprs = adata.to_df()
cd4na_exprs.index = adata.obs['barcodes']
cd4na_exprs = cd4na_exprs.sort_index()


In [45]:
cert_exprs.shape

(49159, 33538)

In [46]:
cd4na_exprs.shape

(164281, 33538)

In [47]:
(cd4na_exprs == cert_exprs).all(axis=1).value_counts()

ValueError: Can only compare identically-labeled (both index and columns) DataFrame objects

In [ ]:
(cd4na_exprs == cert_exprs).all(axis=0).value_counts()


In [66]:
### test with randomization
# find share barcodes in all 3 datase
share_barcodes = set(adata_combined.obs['barcodes']) & set(adata.obs['barcodes'])
share_genes = set(adata_combined.var_names) & set(adata.var_names) 

# select 10000 cells randomly
random_barcodes = random.sample(sorted(share_barcodes), k=10000)

In [67]:
# get test data
joint_adata_test = adata_combined[adata_combined.obs['barcodes'].isin(random_barcodes), 
    adata_combined.var_names.isin(share_genes)].to_memory()
cd4na_adata_test = adata[adata.obs['barcodes'].isin(random_barcodes), 
    adata.var_names.isin(share_genes)].to_memory()
    

In [68]:
# move raw counts to X
joint_adata_test = joint_adata_test.raw.to_adata()
cd4na_adata_test= cd4na_adata_test.raw.to_adata()

In [69]:
cert_exprs = joint_adata_test.to_df()
cert_exprs.index=joint_adata_test.obs['barcodes']
cert_exprs = cert_exprs.sort_index()

In [70]:
cd4na_exprs = cd4na_adata_test.to_df()
cd4na_exprs.index = cd4na_adata_test.obs['barcodes']
cd4na_exprs = cd4na_exprs.sort_index()

In [71]:
(cd4na_exprs.index == cert_exprs.index).all()


True

In [72]:
(cd4na_exprs.columns == cert_exprs.columns).all()

True

In [73]:
(cd4na_exprs == cert_exprs).all(axis=0).value_counts()


True    33538
Name: count, dtype: int64

In [74]:
(cd4na_exprs == cert_exprs).all(axis=1).value_counts()

True    10000
Name: count, dtype: int64

In [48]:
out_file = 'output/preRA_dc_sample_selection_combined_adata_{d}.h5ad'.format(
        d = date.today()
    )
out_file

'output/preRA_dc_sample_selection_combined_adata_2024-06-25.h5ad'

In [49]:
### save
adata_combined.write_h5ad(out_file)

## Upload Cell Type data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [58]:
study_space_uuid = '223de760-9624-45bd-aefe-ca24c75b1800'
title = '07 PBMC L3 Sample Selection Combined Anndata {d}'.format(d = date.today())
title

'07 PBMC L3 Sample Selection Combined Anndata 2024-06-25'

In [59]:
search_id = element_id()
search_id

'neutron-magnesium-beryllium'

In [60]:
in_files = list(h5ad_uuids.values())
in_files

['2558b706-8cac-41c6-bd21-8493d514762d',
 '61222f54-586a-44a8-8aed-61576c1ab381',
 'fe955a14-15e0-4917-9457-814bde477e85',
 '415df319-8ecd-440b-9186-30e5b73f4d7c',
 '62f3b6b8-ae35-47d4-832b-5c91e0b0257f',
 'bd35e899-3b6b-42ec-b7fd-503036ef4be8',
 'fcf2b06d-6fdf-4846-9793-cc2edb92c8ce',
 '8d3dbec0-2179-493a-bf4a-5036a7adae10',
 'e5b5ca1d-9bb3-4afe-ad41-efc778ed292d',
 '75a5fa47-601b-49d1-af1c-b4da9b08a875',
 'a88f617b-1ed8-4093-92ef-f534c3a69b9c',
 'e7780797-a1e1-4760-89db-38aae9ee7a0e',
 '97434b4e-177b-4382-b41b-5f9454ba1e8b',
 '079385a9-c2f6-40c1-be19-dd91e9d1c377',
 'ca1dde4e-5c77-4cc1-b23e-a7065ecebbda',
 'a7e61f28-74c5-45a7-9bfe-dd89524be3e4',
 '4b21b598-2af5-40b0-b194-929356f0481a',
 'bf5ffa30-1ff6-487d-b440-f8dc959d6339',
 '92afa909-08af-4f07-a7e3-20e9aa1a8d75',
 'a2214543-ea76-4872-83fc-1942476a9eb7',
 'e49b3873-eed5-49bc-b1a7-34f29930ca5d',
 '7c347c97-7789-444d-bbef-af324f619ea1',
 '546b9825-a122-4ef6-bea9-fda97c6af2f8',
 'b0928845-28b4-43a5-af53-50e48dadb860',
 'a652b11f-0f55-

In [61]:
out_file

'output/preRA_dc_sample_selection_combined_adata_2024-06-25.h5ad'

In [62]:
hisepy.upload.upload_files(
    files = [out_file],
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

you are trying to upload file_ids... ['output/preRA_dc_sample_selection_combined_adata_2024-06-25.h5ad']. Do you truly want to proceed?


(y/n) y


{'trace_id': '99689fea-3689-42e6-83eb-cfaaceb5c583',
 'files': ['output/preRA_dc_sample_selection_combined_adata_2024-06-25.h5ad']}

In [95]:
import session_info
session_info.show()